# snowflakeR Quickstart -- Workspace Notebook

This notebook is for **Snowflake Workspace Notebooks** (Python kernel + `%%R` magic).
For local environments (RStudio, Posit, JupyterLab), use `local_quickstart.ipynb`.

**Before you start:** Copy `notebook_config.yaml.template` to `notebook_config.yaml`
and edit it with your warehouse, database, and schema.

**Sections:**
1. Setup (install R + snowflakeR)
2. Connect & set execution context
3. Queries & Table Operations
4. Additional Query Patterns
5. Visualization with ggplot2
6. Cleanup

## 1. Setup

Steps 1-2 are the only Python cells you'll need to run. They bootstrap the R
environment, register the `%%R` magic, and install snowflakeR -- after that,
everything is pure R.

### Step 1: Install R environment (~3 minutes, first time only)

Uses the `sfnb-multilang` toolkit to install R and required packages into
user-space via micromamba + conda-forge. No `sudo` or root required.

**Configuration options:** The `install()` call below uses keyword arguments
for a quick start. For more advanced configurations (additional conda/CRAN
packages, ADBC, DuckDB, Scala/Julia, etc.) you can pass a YAML config file:

```python
install(config="configs/snowflaker.yaml")
```

See the preset configs in the
[snowflake-notebook-multilang](https://github.com/Snowflake-Labs/snowflake-notebook-multilang)
repo under `configs/`:
- `configs/snowflaker.yaml` -- R packages for snowflakeR (this package)
- `configs/rsnowflake.yaml` -- R packages for the RSnowflake DBI driver (includes ADBC)
- `configs/r_only.yaml` -- General R with ADBC + DuckDB
- `configs/default.yaml` -- All languages with full options

In [ ]:
# Install R environment and register %%R magic
from sfnb_setup import install_r
install_r(config="snowflaker_config.yaml")

### Step 2: Install and load snowflakeR

Install the `snowflakeR` R package from GitHub (latest version).

In [ ]:
from sfnb_setup import install_r_packages
install_r_packages(config="snowflaker_config.yaml", packages=["snowflakeR"])

---
## 2. Connect & Set Execution Context

From here on, it's all R. No more Python.

Workspace Notebooks do **not** auto-set database or schema.
`sfr_load_notebook_config()` reads `notebook_config.yaml` and runs
`USE WAREHOUSE / DATABASE / SCHEMA` to set the execution context.

All table references in this notebook use fully qualified names via `sfr_fqn()`.

**Tip:** Need to install additional R packages? Use this pattern to suppress
prompts and noise (Workspace Notebooks have no stdin):
```r
options(repos = c(CRAN = "https://cloud.r-project.org"))
install.packages("mypackage", type = "source", quiet = TRUE)
suppressPackageStartupMessages(library(mypackage))
```
For packages with system library dependencies (e.g., `sf`, `curl`), use the
conda-forge equivalent in `r_packages.yaml` instead.

In [ ]:
%%R
library(snowflakeR)

# Connect (auto-detects Workspace session)
conn <- sfr_connect()

# Load config and set execution context
conn <- sfr_load_notebook_config(conn)
conn

---
## 3. Queries & Table Operations

In [ ]:
%%R
# Run a SQL query
result <- sfr_query(conn, "SELECT CURRENT_TIMESTAMP() AS now, CURRENT_USER() AS user_name")
result

In [ ]:
%%R
# Write a data.frame to Snowflake (fully qualified name)
sfr_write_table(conn, sfr_fqn(conn, "SFR_MTCARS"), mtcars, overwrite = TRUE)

In [ ]:
%%R
# List tables
tables <- sfr_list_tables(conn)
rcat("Tables:", paste(head(tables, 10), collapse = ",\n  "))

In [ ]:
%%R
# Read it back (fully qualified name)
df <- sfr_read_table(conn, sfr_fqn(conn, "SFR_MTCARS"))
rview(df, n = 5)

In [ ]:
%%R
# Describe columns
sfr_list_fields(conn, sfr_fqn(conn, "SFR_MTCARS"))

---
## 4. Additional Query Patterns

`sfr_query()` returns data.frames. For standard DBI-compliant access and
`dbplyr`/`dplyr` integration, install the companion
[RSnowflake](https://github.com/Snowflake-Labs/RSnowflake) package. See
`RSnowflake/inst/notebooks/demo_rsnowflake.ipynb` for a full walkthrough
(DBI, dbplyr, Arrow, ADBC). Locally, `local_quickstart.ipynb` Section 4 shows
how to bridge via `sfr_dbi_connection()`.

In [ ]:
%%R
# sfr_query returns results as data.frames
sfr_query(conn, "SELECT 42 AS answer") |> rprint()

# Check table existence
sfr_table_exists(conn, sfr_fqn(conn, "SFR_MTCARS"))

In [ ]:
%%R
# Aggregate query via sfr_query
summary <- sfr_query(conn, paste(
  "SELECT CYL, COUNT(*) AS n, AVG(MPG) AS avg_mpg, AVG(HP) AS avg_hp",
  "FROM", sfr_fqn(conn, "SFR_MTCARS"),
  "GROUP BY CYL ORDER BY CYL"
))

summary

# For dplyr/dbplyr integration, install RSnowflake and use:
#   dbi_con <- sfr_dbi_connection(conn)
#   tbl(dbi_con, I(sfr_fqn(conn, "SFR_MTCARS"))) |>
#     group_by(CYL) |> summarise(n = n(), avg_mpg = mean(MPG)) |> collect()

---
## 5. Visualization with ggplot2

Use `%%R -w WIDTH -h HEIGHT` and `print(p)` for plots in Workspace Notebooks.

In [ ]:
%%R -w 700 -h 450
library(ggplot2)

df <- sfr_read_table(conn, sfr_fqn(conn, "SFR_MTCARS"))

p <- ggplot(df, aes(x = WT, y = MPG, color = factor(CYL))) +
  geom_point(size = 3) +
  labs(title = "MPG vs Weight by Cylinder Count",
       x = "Weight (1000 lbs)", y = "Miles per Gallon",
       color = "Cylinders") +
  theme_minimal()

print(p)  # print() required in Workspace Notebooks

---
## 6. Cleanup

In [ ]:
%%R
# Uncomment to clean up demo objects
# (commented out to avoid accidental deletion on Run All)
#
# sfr_execute(conn, paste("DROP TABLE IF EXISTS", sfr_fqn(conn, "SFR_MTCARS")))
# rcat("Cleanup complete.")